# Phase 3: Face Recognition Pipeline with InsightFace
Notebook ini mendemonstrasikan tahapan dalam pipeline *Face Recognition* menggunakan InsightFace:
1. **Detection:** Deteksi wajah menggunakan SCRFD.
2. **Alignment:** Estimasi *landmarks* 2D.
3. **Embedding:** Ekstraksi fitur (512-dimensi) menggunakan model ArcFace.
4. **Matching:** Pencocokan identitas dengan *Cosine Similarity*.

In [ ]:
import sys
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from backend.app.models.face_engine import FaceEngine

def show_img(img_bgr, title="Image"):
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

### Inisialisasi Face Engine

In [ ]:
# Inisialisasi InsightFace (menggunakan model 'buffalo_l')
engine = FaceEngine.get_instance()
engine.initialize(ctx_id=0) # ctx_id=0 untuk GPU, -1 untuk CPU

### Deteksi Wajah dan Ekstraksi Fitur
Silakan tempatkan sebuah gambar di `data/sample_face.jpg` untuk menjalankan blok ini.

In [ ]:
sample_img_path = os.path.join(project_root, 'data', 'sample_face.jpg')

if os.path.exists(sample_img_path):
    img = cv2.imread(sample_img_path)
    faces = engine.detect_faces(img)
    
    img_draw = img.copy()
    for face in faces:
        bbox = face['bbox']
        cv2.rectangle(img_draw, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
        
        # Draw landmarks
        if face['landmarks']:
            for pt in face['landmarks']:
                cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                
    show_img(img_draw, f"Detected {len(faces)} face(s)")
    
    if len(faces) > 0:
        print(f"Embedding shape: {faces[0]['embedding'].shape}")
else:
    print(f"File not found: {sample_img_path}. Please add a test image.")